# Noveria History — Kokoro Batch Narration
Run these cells interactively in Google Colab. No public API or tunnel is started. Use **bm_george** (British male) and download a ZIP of WAVs for MEP Video Maker.
Free Colab runtime availability is not guaranteed. Keep notebooks private if scripts are private.


In [ ]:
!pip -q install 'kokoro>=0.9.4' soundfile
!apt-get -qq -y install espeak-ng > /dev/null 2>&1
from kokoro import KPipeline
import numpy as np, soundfile as sf, io, zipfile, json, re, os
from google.colab import files
pipeline = KPipeline(lang_code='b')
VOICE = 'bm_george'
SPEED = 0.95
print('Ready:', VOICE)


## Upload scripts
Upload a UTF-8 **JSON file** containing an array of objects like `[{"title":"Western Front","text":"Your full narration..."}]`. The editor's **Download Narration Batch JSON** button creates this format. The notebook generates one WAV per entry.


In [ ]:
uploaded = files.upload()
filename = next((name for name in uploaded if name.lower().endswith('.json')), None)
assert filename, 'Upload a .json batch file'
items = json.loads(uploaded[filename].decode('utf-8'))
assert isinstance(items,list) and 1 <= len(items) <= 100
print('Scripts:', len(items))


In [ ]:
output = io.BytesIO()
with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED) as archive:
    for idx,item in enumerate(items,1):
        title = re.sub(r'[^A-Za-z0-9_-]+','-',str(item.get('title','short')))[:55].strip('-') or 'short'
        text = str(item.get('text','')).strip()
        if not text: continue
        pieces = [audio.detach().cpu().numpy() for _,_,audio in pipeline(text, voice=VOICE, speed=SPEED) if audio is not None]
        if not pieces: continue
        audio = np.concatenate(pieces)
        wav = io.BytesIO()
        sf.write(wav,audio,24000,format='WAV')
        archive.writestr(f'{idx:02d}-{title}.wav',wav.getvalue())
        print(idx,title,round(len(audio)/24000,1),'seconds')
output.seek(0)
with open('noveria-narrations.zip','wb') as f:f.write(output.read())
files.download('noveria-narrations.zip')
